In [ ]:
# 설치

!pip install requests 
!pip install beautifulsoup4 
!pip install pandas 
!pip install numpy

In [ ]:
# 각 라이브러리 별 버전 확인
import requests
import bs4
import pandas as pd
import numpy as np

print("requests ver.:", requests.__version__)
print("beautifulsoup4 ver.:", bs4.__version__)
print("pandas ver.:", pd.__version__)
print("numpy ver.:", np.__version__)


# 혹은 전체 출력 
!pip list

requests ver.: 2.32.5
beautifulsoup4 ver.: 4.13.5
pandas ver.: 2.3.3
numpy ver.: 2.3.5
Package                           Version
--------------------------------- --------------------
absl-py                           2.4.0
aiobotocore                       2.25.0
aiodns                            3.5.0
aiohappyeyeballs                  2.6.1
aiohttp                           3.13.2
aioitertools                      0.12.0
aiosignal                         1.4.0
alabaster                         0.7.16
altair                            5.5.0
anaconda-anon-usage               0.7.5
anaconda-auth                     0.12.3
anaconda-catalogs                 0.2.0
anaconda-cli-base                 0.7.0
anaconda-client                   1.14.0
anaconda-navigator                2.7.0
anaconda-project                  0.11.1
annotated-types                   0.6.0
anyio                             4.10.0
appdirs                           1.4.4
archspec                          0.2.5
argon2-c

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

keyword = "IT"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.naver.com"
}

titles, contents = [], []

for page in range(1, 10 + 1):
    print(f"{page}페이지 저장중...")

    start = (page - 1) * 10 + 1
    url = f"https://search.naver.com/search.naver?where=news&query={keyword}&start={start}"
    
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "lxml")

    news_items = soup.select(".fender-ui_228e3bd1.OhDwxoWO07Yo2tjcNotM")
    success_count = 0

    for item in news_items:
        title = item.get_text().strip()
        titles.append(title)

        # 본문 크롤링 시도
        try:
            # 검색 결과에서 직접 기사 링크 가져오기
            link = item.get("href")  
            res = requests.get(link, headers=headers, timeout=5)
            soup_detail = BeautifulSoup(res.text, "lxml")

            # 본문 추출 시도
            content_tag = soup_detail.select_one("#newsct_article, #dic_area, #articleBodyContents")
            if not content_tag:
                candidates = [
                    "article", ".article_body", ".art_body", "#article_body", 
                    ".news_body", "#articleBody", ".viewer", ".article_view"
                ]
                for cand in candidates:
                    found = soup_detail.select_one(cand)
                    if found:
                        content_tag = found
                        break

            if content_tag:
                text = content_tag.get_text(separator=' ', strip=True)
                contents.append(text)
                success_count += 1
            else:
                contents.append("본문 미식별")

        except Exception as e:
            contents.append(f"접속 실패: {e}")

        time.sleep(0.8)

    # 페이지별 CSV 저장
    df = pd.DataFrame({"title": titles, "content": contents})
    df.to_csv(f"naver_news.csv", index=False, encoding="utf-8-sig")
    print(f"{page}페이지 저장완료! 성공 {success_count} / 전체 {len(news_items)}")

print("전체 수집 완료!")

1페이지 저장중...
1페이지 저장완료! 성공 9 / 전체 10
2페이지 저장중...
2페이지 저장완료! 성공 8 / 전체 10
3페이지 저장중...
3페이지 저장완료! 성공 6 / 전체 10
4페이지 저장중...
4페이지 저장완료! 성공 9 / 전체 10
5페이지 저장중...
5페이지 저장완료! 성공 6 / 전체 10
6페이지 저장중...
6페이지 저장완료! 성공 6 / 전체 10
7페이지 저장중...
7페이지 저장완료! 성공 6 / 전체 10
8페이지 저장중...
8페이지 저장완료! 성공 7 / 전체 10
9페이지 저장중...
9페이지 저장완료! 성공 9 / 전체 10
10페이지 저장중...
10페이지 저장완료! 성공 8 / 전체 10
전체 수집 완료!


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

keyword = "AI"
headers = {
  'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
	'Referer' : 'https://www.naver.com/'
}


news_data = []


for page in range(1, 11):
    start = (page - 1) * 10 + 1
    url = f"https://search.naver.com/search.naver?where=news&query={keyword}&start={start}"
    

    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        print(f"[성공] {page}페이지 저장중. (상태코드: {response.status_code})")
    else:
        print(f"[실패] {page}페이지 저장실패. (상태코드: {response.status_code})")
        continue


    soup = BeautifulSoup(response.text, "lxml")

    news_items = soup.select(".fender-ui_228e3bd1.OhDwxoWO07Yo2tjcNotM")

    for item in news_items:
        title = item.get_text().strip()
        link = item.get("href")
        
        try:
            res_detail = requests.get(link, headers=headers, timeout=5)
            if res_detail.status_code == 200:
                soup_detail = BeautifulSoup(res_detail.text, "lxml")
                
                content_tag = soup_detail.select_one("#newsct_article, #dic_area, #articleBodyContents")
                
                if not content_tag:
                    candidates = ["article", ".article_body", "#articleBody", ".viewer"]
                    for cand in candidates:
                        found = soup_detail.select_one(cand)
                        if found:
                            content_tag = found
                            break
                
                content = content_tag.get_text(separator=' ', strip=True) if content_tag else "내용 저장 실패"
            else:
                content = "본문 페이지 접속 불가"
                
        except Exception as e:
            content = f"에러 발생: {e}"

        news_data.append({"제목": title, "내용": content})
        time.sleep(0.3)

df = pd.DataFrame(news_data)
file_name = "naver_news.csv"
df.to_csv(file_name, index=False, encoding="utf-8-sig")

print("-" * 30)
print(f"전체 수집 완료! 총 {len(df)}건의 데이터가 '{file_name}'로 저장되었습니다.")

[성공] 1페이지 저장중. (상태코드: 200)
[성공] 2페이지 저장중. (상태코드: 200)
[성공] 3페이지 저장중. (상태코드: 200)
[성공] 4페이지 저장중. (상태코드: 200)
[성공] 5페이지 저장중. (상태코드: 200)
[성공] 6페이지 저장중. (상태코드: 200)
[성공] 7페이지 저장중. (상태코드: 200)
[성공] 8페이지 저장중. (상태코드: 200)
[성공] 9페이지 저장중. (상태코드: 200)
[성공] 10페이지 저장중. (상태코드: 200)
------------------------------
전체 수집 완료! 총 100건의 데이터가 'naver_news.csv'로 저장되었습니다.


In [ ]:
import pandas as pd

df = pd.read_csv('naver_news.csv')

df.dropna(inplace=True)

# 제목이 5자 이하인 뉴스 삭제
df = df[df['제목'].str.len() > 5]

print(f"총 {len(df)}건의 뉴스가 전처리 완료.")
print(df.head())

file_name = 'clean_news.csv'

df.to_csv(file_name, index=False, encoding="utf-8-sig")

print(f'{file_name} 파일 저장 완료')



총 100건의 뉴스가 전처리 완료.
                                             제목  \
0           "ERP에 AI 적용" SAP와 손잡은 LG CNS의 AX 전략   
1            메타, 초지능팀 첫 AI모델 발표…오픈AI·구글 경쟁작에 필적   
2  [단독] 네이버클라우드, AI 보안전략 조직 신설… 수장에 국정원 출신 김...   
3              "배탈 났나" 병원찾은 어르신…AI가 살펴보니 '심근경색'   
4               성인 문해교육기관 446곳 선정…AI·디지털 역량 돕는다   

                                                  내용  
0  'SAP 비즈니스 AI' 기반 ERP AX 실행 전략 공개 전문인력 육성…올해 초 ...  
1  메타가 지난해 전방위 인재 영입을 통해 구성한 연구팀이 주요 인공지능 경쟁사 모델에...  
2                                           내용 저장 실패  
3  대웅제약 "AI 분석 통해 급성 심근경색 발견" [서울=뉴시스] 조영욱 운암한국병원...  
4  교육부, '2026년 성인 문해교육 지원사업' 성인 문해교육기관, 전년보다 33곳 ...  
clean_news.csv 파일 저장 완료
